In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from pysr import PySRRegressor
import matplotlib.pyplot as plt

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:
df = pd.read_excel("./banco de dados daniel.xlsx", "Dataset com TS variável")
df = df[df["w (mm)"] <= 0.4]
df = df.drop("Autor", axis=1)
X = df.drop("w (mm)", axis=1)
y = df["w (mm)"]

In [3]:
X = X.rename(columns=lambda c: c.encode("ascii", "ignore").decode("ascii"))
X.columns = [c.replace("(", "").replace(")", "").replace(" ", "_") for c in X.columns]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=33)

In [ ]:
model = PySRRegressor(
    parsimony=0.015,
    maxsize=20,
    maxdepth=6,
    population_size=50,
    ncycles_per_iteration=4000,
    niterations=500,
    adaptive_parsimony_scaling=100.0,
    deterministic=False,
    parallelism='multithreading',
    progress=True,
    should_optimize_constants=True,
    optimizer_algorithm="BFGS",
    optimizer_nrestarts=5,
    binary_operators=["+", "-", "*", "/", "pow"],
    weight_optimize=0.005,
    verbosity=1,
    constraints={"pow": (0, 3)}
)

model.fit(X_train, y_train)

/home/andre/programing/lamfo/ic_andre_rimes/env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...
/home/andre/programing/lamfo/ic_andre_rimes/env/lib/python3.10/site-packages/pysr/sr.py:96: UserWarning: You are using the `^` operator, but have not set up `constraints` for it. This may lead to overly complex expressions. One typical constraint is to use `constraints={..., '^': (-1, 1)}`, which will allow arbitrary-complexity base (-1) but only powers such as a constant or variable (1). For more tips, please see https://ai.damtp.cam.ac.uk/pysr/tuning/
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 5.480e+05
Progress: 195 / 15500 total iterations (1.258%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.474e-02  0.000e+00  y = 0.26695
3           1.204e-02  1.015e-01  y = Tenso_na_armadura_menos_tension_stiffening_MPa * 0.000...
                                      95397
5           1.024e-02  8.090e-02  y = 0.10863 + (Tenso_na_armadura_menos_tension_stiffening_...
                                      MPa * 0.00060317)
7           7.897e-03  1.298e-01  y = 0.0021286 * (Distncia_ao_centro_de_gravidade_da_armadu...
                                      ra__mm + (Tenso_na_armadura_menos_tension_stiffening_MPa /...
                                       Resistncia__trao_do_concreto_MPa))
9           6.699e-03  8.228e-02  y = ((Tenso_na_armadu

In [ ]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)

best_eq = model.get_best()
best_equation = best_eq['equation']

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.scatter(y_train, y_pred_train, alpha=0.6)
ax1.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
ax1.set_xlabel('Actual')
ax1.set_ylabel('Predicted')
ax1.set_title(f'Training Set (R² = {r2_train:.4f})')

ax2.scatter(y_test, y_pred_test, alpha=0.6)
ax2.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
ax2.set_xlabel('Actual')
ax2.set_ylabel('Predicted')
ax2.set_title(f'Test Set (R² = {r2_test:.4f})')

plt.tight_layout()
plt.show()

In [ ]:
best_eq = model.get_best()
print(f"Best equation: {best_eq['equation']}")
print(f"Complexity: {best_eq['complexity']} nodes")

In [ ]:
import sympy as sp
from IPython.display import display, Latex

equation_sympy = model.sympy()
equation_latex = sp.latex(equation_sympy)

table_latex = f"""
\\begin{{array}}{{|l|c|}}
\\hline
\\textbf{{Metric}} & \\textbf{{Value}} \\\\
\\hline
\\text{{Equation}} & {equation_latex} \\\\
\\hline
R^2 \\text{{ (Train)}} & {r2_train:.4f} \\\\
\\hline
R^2 \\text{{ (Test)}} & {r2_test:.4f} \\\\
\\hline
\\text{{RMSE (Train)}} & {rmse_train:.4f} \\\\
\\hline
\\text{{RMSE (Test)}} & {rmse_test:.4f} \\\\
\\hline
\\end{{array}}
"""

display(Latex(table_latex))

In [ ]:
models_pred_df = pd.read_excel("banco de dados daniel.xlsx", "Modelos normativos e empíricos")
y_pred_total = model.predict(X)

models_pred_df["Symbolic Regression"] = None  
for i in range(len(y_pred_total)): 
    models_pred_df.at[i, "Symbolic Regression"] = y_pred_total[i]

models_pred_df["Symbolic Regression"] = pd.to_numeric(
    models_pred_df["Symbolic Regression"], errors="coerce"
)

wexp_col = "wexp (mm)"
for col in models_pred_df.select_dtypes(include="number").columns:
    if col == wexp_col:
        continue
    models_pred_df[f"{col} / wexp"] = models_pred_df[col] / models_pred_df[wexp_col]

models_pred_df.drop(columns=['wexp (mm)', 'fib Model Code 2010', 'fib Model Code 2020', 'Eurocode 2',
       'NBR 6118 - wk1 - fissuração estabilizada',
       'NBR 6118 - wk2 - fissuração não estabilizada', 'Clark (1956)',
       'Leonhardt (1977)', 'Desayi e Ganesan (1985)', 'Oh e Kang (1987)',
       'Caldas (1997)', 'Gergely e Lutz (1968)', ' b  variável e < 0,6',
       'equação (15) do artigo de Belarbi e Hsu (1994)',
       'Beta conforme Fields e Bischoff (2004)', 'slip de Yuan et al. (2025)',
       'slip de Biscaia e Soares (2020)', 'HistGradientBoosting_TS_var',
       'LGBM_TS_var', 'GradientBoosting_TS_var', 'HistGradientBoosting_sem_TS',
       'LGBM_sem_TS', 'GradientBoosting_sem_TS', 'Symbolic Regression'], inplace=True)

numeric_df = models_pred_df.select_dtypes(include="number")
desc = numeric_df.describe().T
desc["skewness"] = numeric_df.skew()
desc["kurtosis"] = numeric_df.kurtosis()
desc_rounded = desc.round(2).drop(columns=["count"])
desc_rounded